# CoT FIRE DSL Experiment

这份 notebook 用于完成 `MiniMind + FIRE DSL` 项目的 CoT 实验，目标是把 reasoning 数据真正转成可训练、可评估的 CoT SFT 流程。

它会覆盖四件事：

1. 检查目录与数据
2. 把 `reasoning_content` 转成真正可训练的 CoT 数据
3. 调用 MiniMind 训练 CoT SFT 模型
4. 同时评估 `raw 输出` 和 `提取 fenced dsl_fire 后的输出`


## 为什么需要先做数据转换

当前项目里的 reasoning 数据中：

- `assistant.content` 只有裸 DSL
- `assistant.reasoning_content` 才是真正的短推理

而标准训练脚本通常只读取 `content`，不会自动读取 `reasoning_content`。
所以如果不先做转换，直接拿 reasoning 数据训练，模型其实学不到 CoT。

因此这里的正确做法是：

- 把 `reasoning_content` 拼回 `assistant.content`
- 最终答案统一写成 fenced `dsl_fire` code block
- 推理时再对输出做提取，避免 reasoning 污染最终评估


In [ ]:
from __future__ import annotations

import csv
import json
import re
import subprocess
from pathlib import Path

ROOT = Path.cwd()
MINIMIND = ROOT / "minimind"
TRAIN_DIR = ROOT / "fire-dsl-data" / "train"
EVAL_DIR = ROOT / "fire-dsl-data" / "eval"
RESULTS_DIR = ROOT / "results"
LOGS_DIR = ROOT / "logs"
EXPERIMENTS_DIR = ROOT / "experiments"

for p in [TRAIN_DIR, EVAL_DIR, RESULTS_DIR, LOGS_DIR, EXPERIMENTS_DIR]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")
print(f"minimind repo: {'OK' if MINIMIND.exists() else 'MISSING'}")


In [ ]:
REASONING_JSONL = TRAIN_DIR / "fire_operator_dsl_sft_reasoning_short.jsonl"
REASONING_AUG_JSONL = TRAIN_DIR / "fire_operator_dsl_sft_reasoning_short_ideas_augmented.jsonl"
CODE_JSONL = TRAIN_DIR / "fire_operator_dsl_sft_code.jsonl"

def read_first_items(path: Path, n: int = 1):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            rows.append(json.loads(line))
            if i + 1 >= n:
                break
    return rows

print("=== reasoning sample ===")
for item in read_first_items(REASONING_JSONL):
    print(json.dumps(item, ensure_ascii=False, indent=2)[:1600])

print("\n=== code sample ===")
for item in read_first_items(CODE_JSONL):
    print(json.dumps(item, ensure_ascii=False, indent=2)[:1600])


## 构造 CoT 训练数据

这里采用的策略是：

- `system`：改成允许先短推理，再给最终 code block
- `user`：明确要求最终输出 fenced `dsl_fire`
- `assistant`：拼成 `推理 + 最终 fenced dsl_fire`

这样训练目标就和你真正想做的 CoT 实验保持一致。


In [ ]:
COT_JSONL = TRAIN_DIR / "fire_operator_dsl_sft_cot_codeblock.jsonl"
COT_AUG_JSONL = TRAIN_DIR / "fire_operator_dsl_sft_cot_codeblock_ideas_augmented.jsonl"

SYSTEM_COT = (
    "你是一个金融量化因子 DSL 代码生成器。"
    "用户会给出自然语言金融想法。"
    "你可以先进行简短推理，但最终答案必须输出一个 Markdown fenced code block。"
    "代码块语言标记必须是 dsl_fire。"
    "代码块内只写一行 result = <合法 FIRE DSL 表达式>。"
    "字段只能使用 open, close, high, low, volume, vwap, pb, market_cap, industry。"
)

def fence_dsl(expr: str) -> str:
    expr = expr.strip()
    expr = re.sub(r"^result\s*=\s*", "", expr)
    return f"```dsl_fire\nresult = {expr}\n```"

def build_assistant_content(reasoning: str, expr: str) -> str:
    reasoning = (reasoning or "").strip()
    final_code = fence_dsl(expr)
    if reasoning:
        return f"推理：\n{reasoning}\n\n最终答案：\n{final_code}"
    return final_code

def rewrite_user(content: str) -> str:
    content = content.strip()
    if "dsl_fire" in content and "代码块" in content:
        return content
    return content + "\n\n请先做简短推理，最终必须输出 fenced `dsl_fire` code block，且代码块内只写一行 `result = <DSL表达式>`。"

def convert_reasoning_jsonl(src: Path, dst: Path):
    count = 0
    dst.parent.mkdir(parents=True, exist_ok=True)
    with src.open("r", encoding="utf-8") as fin, dst.open("w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip():
                continue
            obj = json.loads(line)
            new_convs = []
            for turn in obj["conversations"]:
                role = turn["role"]
                if role == "system":
                    turn = {**turn, "content": SYSTEM_COT, "reasoning_content": ""}
                elif role == "user":
                    turn = {**turn, "content": rewrite_user(turn.get("content", "")), "reasoning_content": ""}
                elif role == "assistant":
                    turn = {
                        **turn,
                        "content": build_assistant_content(turn.get("reasoning_content", ""), turn.get("content", "")),
                        "reasoning_content": "",
                    }
                new_convs.append(turn)
            obj["conversations"] = new_convs
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            count += 1
    return count

base_count = convert_reasoning_jsonl(REASONING_JSONL, COT_JSONL)
aug_count = convert_reasoning_jsonl(REASONING_AUG_JSONL, COT_AUG_JSONL)
print("written:", COT_JSONL, base_count)
print("written:", COT_AUG_JSONL, aug_count)

with COT_JSONL.open("r", encoding="utf-8") as f:
    sample = json.loads(next(f))
print(json.dumps(sample, ensure_ascii=False, indent=2)[:2200])


## 训练配置

推荐优先跑 `CoT from domain pretrain`，也就是从你们当前最有效的 `fire_dsl_domain_pretrain` 出发。
如果想做更严格 ablation，可以把 `FROM_WEIGHT` 改成 `none`。


In [ ]:
RUN_NAME = "fire_dsl_cot_sft"
FROM_WEIGHT = "fire_dsl_domain_pretrain"
TRAIN_DATA = str(COT_JSONL)
HIDDEN_SIZE = 128
NUM_LAYERS = 2
MAX_SEQ_LEN = 512
BATCH_SIZE = 2
LEARNING_RATE = 3e-4
EPOCHS = 3
DEVICE = "cuda:0"
DTYPE = "float16"
RUN_TRAIN = False

def run_cmd(cmd: str, cwd: Path | None = None):
    print(cmd)
    completed = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")

if not MINIMIND.exists():
    print("WARNING: minimind/ 目录不存在。请先把 MiniMind 源码准备到项目根目录。")

train_cmd = f'''python train_full_sft.py \
  --data_path {TRAIN_DATA} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --learning_rate {LEARNING_RATE} \
  --num_workers 0 \
  --device {DEVICE} \
  --dtype {DTYPE} \
  --hidden_size {HIDDEN_SIZE} \
  --num_hidden_layers {NUM_LAYERS} \
  --max_seq_len {MAX_SEQ_LEN} \
  --from_weight {FROM_WEIGHT} \
  --save_dir ../out \
  --save_weight {RUN_NAME}'''

print(train_cmd)
if RUN_TRAIN:
    run_cmd(train_cmd + f" 2>&1 | tee ../../logs/train_{RUN_NAME}.log", cwd=MINIMIND / "trainer")


## 评估 CoT 模型

这里把评估分成两种：

- `raw`：直接用模型原始输出打分
- `extract`：只提取第一个 fenced `dsl_fire` code block 后再打分

这一步是 CoT 实验的关键，因为它能区分：

- CoT 是否真的没用
- 还是 CoT 有帮助，但被格式污染掩盖了


In [ ]:
EVAL_RUN_NAME = RUN_NAME
WEIGHT = RUN_NAME
EVAL_SET = "generalization"
TEMPERATURE = 0.01
MAX_NEW_TOKENS = 192
TOP_P = 0.95
DEVICE_EVAL = "cuda"
RUN_EVAL = False
RUN_SCORE = False

EVAL_JSONL = EVAL_DIR / f"fire_operator_dsl_eval_code_{EVAL_SET}.jsonl"
RAW_PRED_CSV = RESULTS_DIR / f"{EVAL_RUN_NAME}_{EVAL_SET}_raw_predictions.csv"
EXTRACT_PRED_CSV = RESULTS_DIR / f"{EVAL_RUN_NAME}_{EVAL_SET}_extract_predictions.csv"
RAW_SCORE_LOG = LOGS_DIR / f"{EVAL_RUN_NAME}_{EVAL_SET}_raw_score.log"
EXTRACT_SCORE_LOG = LOGS_DIR / f"{EVAL_RUN_NAME}_{EVAL_SET}_extract_score.log"
RAW_ERROR_CSV = RESULTS_DIR / f"{EVAL_RUN_NAME}_{EVAL_SET}_raw_errors.csv"
EXTRACT_ERROR_CSV = RESULTS_DIR / f"{EVAL_RUN_NAME}_{EVAL_SET}_extract_errors.csv"

infer_cmd = f'''python scripts/batch_fire_infer.py \
  --eval_jsonl {EVAL_JSONL} \
  --output_csv {RAW_PRED_CSV} \
  --save_dir minimind/out \
  --weight {WEIGHT} \
  --hidden_size {HIDDEN_SIZE} \
  --num_hidden_layers {NUM_LAYERS} \
  --device {DEVICE_EVAL} \
  --temperature {TEMPERATURE} \
  --max_new_tokens {MAX_NEW_TOKENS} \
  --top_p {TOP_P}'''

print(infer_cmd)
if RUN_EVAL:
    run_cmd(infer_cmd + f" 2>&1 | tee {LOGS_DIR / (EVAL_RUN_NAME + '_' + EVAL_SET + '_infer.log')}", cwd=ROOT)

def extract_first_dsl_block(text: str) -> str:
    text = (text or "").strip()
    m = re.search(r"```dsl_fire\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        body = m.group(1).strip()
        if not body.startswith("result ="):
            body = "result = " + re.sub(r"^result\s*=\s*", "", body)
        return f"```dsl_fire\n{body}\n```"
    m = re.search(r"result\s*=\s*(.+)", text, flags=re.DOTALL)
    if m:
        expr = m.group(1).strip().split("```", 1)[0].strip()
        return f"```dsl_fire\nresult = {expr}\n```"
    return text

def make_extracted_csv(src_csv: Path, dst_csv: Path):
    rows = []
    with src_csv.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        for row in reader:
            row = dict(row)
            row["prediction"] = extract_first_dsl_block(row.get("prediction", ""))
            rows.append(row)
    with dst_csv.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return len(rows)

if RAW_PRED_CSV.exists():
    n = make_extracted_csv(RAW_PRED_CSV, EXTRACT_PRED_CSV)
    print(f"wrote {n} extracted rows -> {EXTRACT_PRED_CSV}")
else:
    print("Raw prediction CSV not found yet. Run inference first.")

def score_and_analyze(pred_csv: Path, score_log: Path, error_csv: Path):
    score_cmd = f"python scripts/score_fire_predictions.py {pred_csv}"
    error_cmd = f"python scripts/analyze_fire_errors.py {pred_csv} {error_csv}"
    run_cmd(score_cmd + f" 2>&1 | tee {score_log}", cwd=ROOT)
    run_cmd(error_cmd, cwd=ROOT)

if RUN_SCORE:
    if RAW_PRED_CSV.exists():
        score_and_analyze(RAW_PRED_CSV, RAW_SCORE_LOG, RAW_ERROR_CSV)
    if EXTRACT_PRED_CSV.exists():
        score_and_analyze(EXTRACT_PRED_CSV, EXTRACT_SCORE_LOG, EXTRACT_ERROR_CSV)


In [ ]:
def parse_score_log(path: Path):
    if not path.exists():
        return None
    text = path.read_text(encoding="utf-8", errors="ignore")
    names = [
        "exact_dsl_match",
        "fenced_code_format",
        "python_parse_ok",
        "fire_dsl_executable",
        "format_error_rows",
        "parse_error_rows",
        "execution_error_rows",
    ]
    row = {"file": path.name}
    for name in names:
        m = re.search(rf"^{re.escape(name)}:\s*([0-9.]+%?)", text, flags=re.MULTILINE)
        row[name] = m.group(1) if m else ""
    return row

rows = []
for p in [RAW_SCORE_LOG, EXTRACT_SCORE_LOG]:
    row = parse_score_log(p)
    if row:
        rows.append(row)
rows


In [ ]:
SUMMARY_CSV = RESULTS_DIR / "summary_table.csv"
if SUMMARY_CSV.exists():
    import pandas as pd
    df = pd.read_csv(SUMMARY_CSV)
    display(df[df["run_name"].astype(str).str.contains("cot", case=False, na=False)])
else:
    print("summary_table.csv not found")


## 推荐执行顺序

1. 先运行“构造 CoT 训练数据”部分
2. 检查转换后的样本是否符合预期
3. 把 `RUN_TRAIN = True`，开始训练 CoT 模型
4. 把 `RUN_EVAL = True`，生成 raw predictions
5. 运行提取逻辑，得到 extracted predictions
6. 把 `RUN_SCORE = True`，分别评估 raw 和 extract
7. 对比 raw vs extract，判断 CoT 是否真正带来语义收益

推荐重点观察：

- `generalization` 上的 `python_parse_ok`
- `generalization` 上的 `fire_dsl_executable`
- `semantic_mismatch` 是否下降

如果 `extract` 明显优于 `raw`，说明 CoT 有帮助，但需要输出约束或后处理配合。
